# GDM, PE, PTB Cost-Effectiveness Analysis
Reads sensitivity/specificity from review notebook CSVs.

GDM model is run synthetically. PE and Preterm analysis is commented out but can be run with same metrics after clinical utility results.

In [1]:
import pandas as pd
import numpy as np

# Load corrected metrics from review notebooks (run those first!)
# These CSVs are produced by the fixed Review notebooks
try:
    gdm_raw = pd.read_csv('/results/GDM_12w_base_metrics.csv')
    gdm_metrics = gdm_raw.rename(columns={
        'threshold_label': 'threshold',
    })[['threshold', 'n_flagged', 'tp', 'sensitivity', 'specificity', 'ppv']]
    print('Loaded GDM metrics from GDM_12w_base_metrics.csv')
    print(gdm_metrics.to_string(index=False))
except FileNotFoundError:
    print('WARNING: GDM_12w_base_metrics.csv not found!')
    print('Run the GDM 12 - Clinical Utility notebook first.')

'''
print()

try:
    ptb_raw = pd.read_csv('PTB_12w_base_metrics.csv')
    ptb_metrics = ptb_raw.rename(columns={
        'threshold_label': 'threshold',
    })[['threshold', 'n_flagged', 'tp', 'sensitivity', 'specificity', 'ppv']]
    print('Loaded PTB metrics from PTB_12w_base_metrics.csv')
    print(ptb_metrics.to_string(index=False))
except FileNotFoundError:
    print('WARNING: PTB_12w_base_metrics.csv not found!')
    print('Run the Preterm 12 - Clinical Utility notebook first.')

print()

try:
    pe_raw = pd.read_csv('PE_12w_threshold_metrics.csv')
    pe_metrics = pe_raw.rename(columns={
        'Threshold': 'threshold',
        'n Flagged': 'n_flagged',
        'True Pos': 'tp',
        'PPV_raw': 'ppv',
    })
    # Parse string percentages back to floats if needed
    for col in ['sensitivity', 'specificity']:
        if col not in pe_metrics.columns:
            # Parse from formatted string columns
            pe_metrics['sensitivity'] = pe_metrics['Sensitivity'].str.rstrip('%').astype(float) / 100
            pe_metrics['specificity'] = pe_metrics['Specificity'].str.rstrip('%').astype(float) / 100
            break
    pe_metrics = pe_metrics[['threshold', 'n_flagged', 'tp', 'sensitivity', 'specificity', 'ppv']]
    print('Loaded PE metrics from PE_12w_threshold_metrics.csv')
    print(pe_metrics.to_string(index=False))
except FileNotFoundError:
    print('WARNING: PE_12w_threshold_metrics.csv not found!')
    print('Run the PE 12 - Clinical Utility notebook first.')
'''

Loaded GDM metrics from GDM_12w_base_metrics.csv
threshold  n_flagged  tp  sensitivity  specificity   ppv
   Top 1%         10   1     0.026316     0.990644 0.100
   Top 5%         50   8     0.210526     0.956341 0.160
  Top 10%        100  13     0.342105     0.909563 0.130
  Top 20%        200  19     0.500000     0.811850 0.095


"\nprint()\n\ntry:\n    ptb_raw = pd.read_csv('PTB_12w_base_metrics.csv')\n    ptb_metrics = ptb_raw.rename(columns={\n        'threshold_label': 'threshold',\n    })[['threshold', 'n_flagged', 'tp', 'sensitivity', 'specificity', 'ppv']]\n    print('Loaded PTB metrics from PTB_12w_base_metrics.csv')\n    print(ptb_metrics.to_string(index=False))\nexcept FileNotFoundError:\n    print('WARNING: PTB_12w_base_metrics.csv not found!')\n    print('Run the Preterm 12 - Clinical Utility notebook first.')\n\nprint()\n\ntry:\n    pe_raw = pd.read_csv('PE_12w_threshold_metrics.csv')\n    pe_metrics = pe_raw.rename(columns={\n        'Threshold': 'threshold',\n        'n Flagged': 'n_flagged',\n        'True Pos': 'tp',\n        'PPV_raw': 'ppv',\n    })\n    # Parse string percentages back to floats if needed\n    for col in ['sensitivity', 'specificity']:\n        if col not in pe_metrics.columns:\n            # Parse from formatted string columns\n            pe_metrics['sensitivity'] = pe_metr

## GDM

In [2]:

# =============================================================================
# GDM COST-EFFECTIVENESS
# =============================================================================

# Parameters
GDM_COST = 15593  # Lenoir-Wijnkoop 2015

GDM_INTERVENTIONS = {
    'lifestyle': {
        'name': 'Lifestyle (diet + exercise)',
        'rr': 0.77,
        'rr_ci_low': 0.69,
        'rr_ci_high': 0.87,
        'cost': 199,
        'source': 'Guo et al. BJOG 2019',
        'analysis_type': 'Primary'
    },
    'myoinositol': {
        'name': 'Myo-inositol (4g/day)',
        'rr': 0.42,
        'rr_ci_low': 0.26,
        'rr_ci_high': 0.67,
        'cost': 100,
        'source': 'Greff et al. Nutrients 2023',
        'analysis_type': 'Sensitivity'
    }
}

def gdm_cost_effectiveness(df_metrics, intervention_key, outcome_cost=GDM_COST):
    intv = GDM_INTERVENTIONS[intervention_key]
    rrr = 1 - intv['rr']
    rrr_low = 1 - intv['rr_ci_high']  # Conservative
    rrr_high = 1 - intv['rr_ci_low']  # Optimistic
    
    results = []
    for _, row in df_metrics.iterrows():
        ppv = row['ppv']
        
        nnt = 1 / (ppv * rrr)
        nnt_low = 1 / (ppv * rrr_high)  # Optimistic (lower NNT)
        nnt_high = 1 / (ppv * rrr_low)  # Conservative (higher NNT)
        
        cost_per = nnt * intv['cost']
        cost_per_low = nnt_low * intv['cost']
        cost_per_high = nnt_high * intv['cost']
        
        net_savings = outcome_cost - cost_per
        net_savings_low = outcome_cost - cost_per_high  # Conservative
        net_savings_high = outcome_cost - cost_per_low  # Optimistic
        
        results.append({
            'Threshold': row['threshold'],
            'n_flagged': int(row['n_flagged']),
            'TP': int(row['tp']),
            'Sensitivity': f"{row['sensitivity']:.1%}",
            'Specificity': f"{row['specificity']:.1%}",
            'PPV': f"{row['ppv']:.1%}",
            'NNT': round(nnt, 1),
            'NNT_range': f"({nnt_low:.1f}-{nnt_high:.1f})",
            'Cost_per_prevented': f"${cost_per:,.0f}",
            'Net_savings': f"${net_savings:,.0f}",
            'Net_savings_range': f"(${net_savings_high:,.0f} to ${net_savings_low:,.0f})",
            'Cost_effective': 'Yes' if net_savings > 0 else 'No'
        })
    
    return pd.DataFrame(results)

# Run GDM analysis
print("="*80)
print("GDM COST-EFFECTIVENESS ANALYSIS")
print("="*80)
print(f"Test set: 78,708 pregnancies | GDM cases: 6,533 | Prevalence: 8.3%")
print(f"GDM incremental cost: ${GDM_COST:,} (Lenoir-Wijnkoop 2015)")

gdm_results = {}
for key, intv in GDM_INTERVENTIONS.items():
    print(f"\n{'-'*80}")
    print(f"[{intv['analysis_type']}] {intv['name']}")
    print(f"RR: {intv['rr']} (95% CI: {intv['rr_ci_low']}-{intv['rr_ci_high']}) | Cost: ${intv['cost']}")
    print(f"Source: {intv['source']}")
    print(f"{'-'*80}")
    
    df_result = gdm_cost_effectiveness(gdm_metrics, key)
    gdm_results[key] = df_result
    print(df_result.to_string(index=False))
    df_result.to_csv(f'/results/GDM_12w_costeff_{key}_updated.csv', index=False)


GDM COST-EFFECTIVENESS ANALYSIS
Test set: 78,708 pregnancies | GDM cases: 6,533 | Prevalence: 8.3%
GDM incremental cost: $15,593 (Lenoir-Wijnkoop 2015)

--------------------------------------------------------------------------------
[Primary] Lifestyle (diet + exercise)
RR: 0.77 (95% CI: 0.69-0.87) | Cost: $199
Source: Guo et al. BJOG 2019
--------------------------------------------------------------------------------
Threshold  n_flagged  TP Sensitivity Specificity   PPV  NNT   NNT_range Cost_per_prevented Net_savings   Net_savings_range Cost_effective
   Top 1%         10   1        2.6%       99.1% 10.0% 43.5 (32.3-76.9)             $8,652      $6,941    ($9,174 to $285)            Yes
   Top 5%         50   8       21.1%       95.6% 16.0% 27.2 (20.2-48.1)             $5,408     $10,185 ($11,581 to $6,026)            Yes
  Top 10%        100  13       34.2%       91.0% 13.0% 33.4 (24.8-59.2)             $6,656      $8,937 ($10,655 to $3,818)            Yes
  Top 20%        200  19

## Preterm Birth Cost-Effectiveness

In [3]:
'''
# Parameters
PTB_COST = 51600  # IOM 2007
SCREENING_COST = 250
PROGESTERONE_COST = 400
CERCLAGE_COST = 2500  # Estimate for sensitivity analysis

SHORT_CERVIX_PREV = 0.02  # ~2% have short cervix ≤25mm

# Screening uptake assumptions
BASELINE_SCREENING = 0.15  # Standard care
MODEL_SCREENING = 0.90     # Model-enhanced

PTB_INTERVENTIONS = {
    'progesterone': {
        'name': 'Vaginal progesterone (CL ≤25mm)',
        'rr': 0.66,
        'rr_ci_low': 0.52,
        'rr_ci_high': 0.83,
        'treatment_cost': PROGESTERONE_COST,
        'source': 'Romero et al. 2016',
        'analysis_type': 'Primary'
    },
    'cerclage': {
        'name': 'Cerclage (prior sPTB + CL <25mm)',
        'rr': 0.70,
        'rr_ci_low': 0.55,
        'rr_ci_high': 0.89,
        'treatment_cost': CERCLAGE_COST,
        'source': 'Berghella et al. 2011',
        'analysis_type': 'Sensitivity'
    }
}

def ptb_cost_effectiveness(df_metrics, intervention_key, 
                           baseline_screening=BASELINE_SCREENING,
                           model_screening=MODEL_SCREENING,
                           short_cervix_prev=SHORT_CERVIX_PREV,
                           outcome_cost=PTB_COST):
    
    intv = PTB_INTERVENTIONS[intervention_key]
    rrr = 1 - intv['rr']
    
    results = []
    for _, row in df_metrics.iterrows():
        n_flagged = row['n_flagged']
        ppv = row['ppv']
        
        # Model arm: enhanced screening uptake
        n_screened_model = n_flagged * model_screening
        n_short_cervix_model = n_screened_model * short_cervix_prev
        cases_prevented_model = n_short_cervix_model * ppv * rrr
        cost_model = (n_screened_model * SCREENING_COST) + (n_short_cervix_model * intv['treatment_cost'])
        
        # Standard care arm: baseline screening
        n_screened_baseline = n_flagged * baseline_screening
        n_short_cervix_baseline = n_screened_baseline * short_cervix_prev
        cases_prevented_baseline = n_short_cervix_baseline * ppv * rrr
        cost_baseline = (n_screened_baseline * SCREENING_COST) + (n_short_cervix_baseline * intv['treatment_cost'])
        
        # Incremental (model vs standard care)
        incremental_cases = cases_prevented_model - cases_prevented_baseline
        incremental_cost = cost_model - cost_baseline
        
        if incremental_cases > 0:
            cost_per_prevented = incremental_cost / incremental_cases
            net_savings = outcome_cost - cost_per_prevented
            cost_eff = 'Yes' if net_savings > 0 else 'No'
        else:
            cost_per_prevented = float('inf')
            net_savings = float('-inf')
            cost_eff = 'N/A'
        
        # Total program value
        total_savings = (incremental_cases * outcome_cost) - incremental_cost
        
        results.append({
            'Threshold': row['threshold'],
            'n_flagged': int(n_flagged),
            'n_screened_incremental': int(n_screened_model - n_screened_baseline),
            'n_treated_incremental': round(n_short_cervix_model - n_short_cervix_baseline, 1),
            'TP': int(row['tp']),
            'Sensitivity': f"{row['sensitivity']:.1%}",
            'PPV': f"{row['ppv']:.1%}",
            'Cases_prevented': round(incremental_cases, 2),
            'Cost_per_prevented': f"${cost_per_prevented:,.0f}" if cost_per_prevented < 1e7 else 'N/A',
            'Net_savings_per_case': f"${net_savings:,.0f}" if abs(net_savings) < 1e7 else 'N/A',
            'Total_program_savings': f"${total_savings:,.0f}",
            'Cost_effective': cost_eff
        })
    
    return pd.DataFrame(results)

# Run PTB analysis
print("\n\n" + "="*80)
print("PTB COST-EFFECTIVENESS ANALYSIS (Triage Model)")
print("="*80)
print(f"Test set: 83,371 pregnancies | PTB cases: 6,800 | Prevalence: 8.2%")
print(f"PTB cost: ${PTB_COST:,} (IOM 2007)")
print(f"CL screening cost: ${SCREENING_COST} | Short cervix prevalence: {SHORT_CERVIX_PREV:.0%}")
print(f"Screening uptake — Baseline: {BASELINE_SCREENING:.0%} | Model-enhanced: {MODEL_SCREENING:.0%}")

ptb_results = {}
for key, intv in PTB_INTERVENTIONS.items():
    print(f"\n{'-'*80}")
    print(f"[{intv['analysis_type']}] {intv['name']}")
    print(f"RR: {intv['rr']} (95% CI: {intv['rr_ci_low']}-{intv['rr_ci_high']}) | Treatment cost: ${intv['treatment_cost']}")
    print(f"Source: {intv['source']}")
    print(f"{'-'*80}")
    
    df_result = ptb_cost_effectiveness(ptb_metrics, key)
    ptb_results[key] = df_result
    print(df_result.to_string(index=False))
    df_result.to_csv(f'PTB_12w_costeff_{key}_triage.csv', index=False)
'''

'\n# Parameters\nPTB_COST = 51600  # IOM 2007\nSCREENING_COST = 250\nPROGESTERONE_COST = 400\nCERCLAGE_COST = 2500  # Estimate for sensitivity analysis\n\nSHORT_CERVIX_PREV = 0.02  # ~2% have short cervix ≤25mm\n\n# Screening uptake assumptions\nBASELINE_SCREENING = 0.15  # Standard care\nMODEL_SCREENING = 0.90     # Model-enhanced\n\nPTB_INTERVENTIONS = {\n    \'progesterone\': {\n        \'name\': \'Vaginal progesterone (CL ≤25mm)\',\n        \'rr\': 0.66,\n        \'rr_ci_low\': 0.52,\n        \'rr_ci_high\': 0.83,\n        \'treatment_cost\': PROGESTERONE_COST,\n        \'source\': \'Romero et al. 2016\',\n        \'analysis_type\': \'Primary\'\n    },\n    \'cerclage\': {\n        \'name\': \'Cerclage (prior sPTB + CL <25mm)\',\n        \'rr\': 0.70,\n        \'rr_ci_low\': 0.55,\n        \'rr_ci_high\': 0.89,\n        \'treatment_cost\': CERCLAGE_COST,\n        \'source\': \'Berghella et al. 2011\',\n        \'analysis_type\': \'Sensitivity\'\n    }\n}\n\ndef ptb_cost_effectivene

## Cost Effectiveness Sensitivity Analyses

In [4]:


# PE cost-effectiveness is primarily computed in the `PE 12 - Clinical Utility` notebook
# (which uses aspirin = $10 and model PE_12w_4-07-26). We replicate the PE math here
# only to populate the summary row using canonical constants.
ASPIRIN_COST = 10
PE_COST = 17500

summary_data = []

# GDM - Lifestyle (Primary)
ppv_gdm = gdm_metrics[gdm_metrics['threshold'] == 'Top 10%']['ppv'].values[0]
rrr_lifestyle = 0.23
nnt_life = 1 / (ppv_gdm * rrr_lifestyle)
cost_life = nnt_life * 199
net_life = GDM_COST - cost_life
summary_data.append({
    'Outcome': 'GDM',
    'Intervention': 'Lifestyle (Primary)',
    'RR': 0.77,
    'NNT': round(nnt_life, 1),
    'Cost_per_prevented': f'${cost_life:,.0f}',
    'Net_savings': f'${net_life:,.0f}',
    'Cost_effective': 'Yes' if net_life > 0 else 'No'
})

# GDM - Myo-inositol (Sensitivity)
rrr_myo = 0.58
nnt_myo = 1 / (ppv_gdm * rrr_myo)
cost_myo = nnt_myo * 100
net_myo = GDM_COST - cost_myo
summary_data.append({
    'Outcome': 'GDM',
    'Intervention': 'Myo-inositol (Sensitivity)',
    'RR': 0.42,
    'NNT': round(nnt_myo, 1),
    'Cost_per_prevented': f'${cost_myo:,.0f}',
    'Net_savings': f'${net_myo:,.0f}',
    'Cost_effective': 'Yes' if net_myo > 0 else 'No'
})

'''
# PTB - Progesterone (Primary) - uses simplified Top 10% scenario with 2% short-cervix baseline.
# See the NuMom2b scenarios cell below for the analysis that drives the manuscript table.
ppv_ptb = ptb_metrics[ptb_metrics['threshold'] == 'Top 10%']['ppv'].values[0]
n_flagged_ptb = int(ptb_metrics[ptb_metrics['threshold'] == 'Top 10%']['n_flagged'].values[0])
incremental_screened = n_flagged_ptb * (MODEL_SCREENING - BASELINE_SCREENING)
incremental_treated = incremental_screened * SHORT_CERVIX_PREV
incremental_cases = incremental_treated * ppv_ptb * 0.34
incremental_cost = (incremental_screened * SCREENING_COST) + (incremental_treated * PROGESTERONE_COST)
cost_ptb = incremental_cost / incremental_cases if incremental_cases > 0 else float('inf')
net_ptb = PTB_COST - cost_ptb
summary_data.append({
    'Outcome': 'PTB',
    'Intervention': 'Triage -> Progesterone (Primary)',
    'RR': 0.66,
    'NNT': round(n_flagged_ptb / incremental_cases, 0) if incremental_cases > 0 else 'N/A',
    'Cost_per_prevented': f'${cost_ptb:,.0f}',
    'Net_savings': f'${net_ptb:,.0f}',
    'Cost_effective': 'Yes' if net_ptb > 0 else 'No'
})

# PE - Aspirin USPSTF (Primary)
ppv_pe = pe_metrics[pe_metrics['threshold'] == 'Top 10%']['ppv'].values[0]
rrr_aspirin = 0.15
nnt_aspirin = 1 / (ppv_pe * rrr_aspirin)
cost_aspirin = nnt_aspirin * ASPIRIN_COST
net_aspirin = PE_COST - cost_aspirin
summary_data.append({
    'Outcome': 'PE',
    'Intervention': 'Low-dose aspirin, USPSTF (Primary)',
    'RR': 0.85,
    'NNT': round(nnt_aspirin, 1),
    'Cost_per_prevented': f'${cost_aspirin:,.0f}',
    'Net_savings': f'${net_aspirin:,.0f}',
    'Cost_effective': 'Yes' if net_aspirin > 0 else 'No'
})

# PE - Aspirin ASPRE (Sensitivity)
rrr_aspre = 0.62
nnt_aspre = 1 / (ppv_pe * rrr_aspre)
cost_aspre = nnt_aspre * ASPIRIN_COST
net_aspre = PE_COST - cost_aspre
summary_data.append({
    'Outcome': 'PE',
    'Intervention': 'Low-dose aspirin, ASPRE (Sensitivity)',
    'RR': 0.38,
    'NNT': round(nnt_aspre, 1),
    'Cost_per_prevented': f'${cost_aspre:,.0f}',
    'Net_savings': f'${net_aspre:,.0f}',
    'Cost_effective': 'Yes' if net_aspre > 0 else 'No'
})

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))
df_summary.to_csv('GDM_PTB_PE_cost_effectiveness_summary.csv', index=False)

print('\n\nAll results saved to CSV files.')
'''



SUMMARY: Cost-Effectiveness at Top 10% Threshold


"\n# PTB - Progesterone (Primary) - uses simplified Top 10% scenario with 2% short-cervix baseline.\n# See the NuMom2b scenarios cell below for the analysis that drives the manuscript table.\nppv_ptb = ptb_metrics[ptb_metrics['threshold'] == 'Top 10%']['ppv'].values[0]\nn_flagged_ptb = int(ptb_metrics[ptb_metrics['threshold'] == 'Top 10%']['n_flagged'].values[0])\nincremental_screened = n_flagged_ptb * (MODEL_SCREENING - BASELINE_SCREENING)\nincremental_treated = incremental_screened * SHORT_CERVIX_PREV\nincremental_cases = incremental_treated * ppv_ptb * 0.34\nincremental_cost = (incremental_screened * SCREENING_COST) + (incremental_treated * PROGESTERONE_COST)\ncost_ptb = incremental_cost / incremental_cases if incremental_cases > 0 else float('inf')\nnet_ptb = PTB_COST - cost_ptb\nsummary_data.append({\n    'Outcome': 'PTB',\n    'Intervention': 'Triage -> Progesterone (Primary)',\n    'RR': 0.66,\n    'NNT': round(n_flagged_ptb / incremental_cases, 0) if incremental_cases > 0 else 

In [5]:
'''
# =============================================================================
# PTB Sensitivity Analysis with Different Short Cervix Prevalence Based on the NuMom2b Study
# =============================================================================

# Load PTB metrics from the base_metrics DataFrame already loaded in the first cell
# (PTB_12w_base_metrics.csv is produced by the Preterm 12 - Clinical Utility notebook).
ptb_data = {
    row['threshold']: {'n_flagged': int(row['n_flagged']), 'ppv': float(row['ppv'])}
    for _, row in ptb_metrics.iterrows()
}
print("PTB base metrics loaded for NuMom2b analysis:")
for k, v in ptb_data.items():
    print(f"  {k}: n_flagged={v['n_flagged']:,}, ppv={v['ppv']:.4f}")

# Parameters
PTB_COST = 51600  # IOM 2007
SCREENING_COST = 250
PROGESTERONE_COST = 400
PROG_RRR = 0.34  # 1 - 0.66

# Screening uptake
BASELINE_SCREENING = 0.15
MODEL_SCREENING = 0.90

# NuMom2b-derived short cervix prevalence scenarios
SHORT_CERVIX_SCENARIOS = {
    'Conservative (Visit 2 population)': 0.025,      # 2.5% - general population at 16-22 wks
    'Moderate (Visit 2 SPTB enriched)': 0.08,        # 8.0% - among women with SPTB <37wk
    'Later screening (Visit 3 population)': 0.072,   # 7.2% - general population at 22-30 wks
    'High enrichment (Visit 3 SPTB)': 0.233,         # 23.3% - among SPTB <37wk at visit 3
}

print("\n" + "="*90)
print("PTB COST-EFFECTIVENESS - NuMom2b Short Cervix Prevalence Scenarios")
print("="*90)
print(f"PTB cost: ${PTB_COST:,} | CL screening: ${SCREENING_COST} | Progesterone: ${PROGESTERONE_COST}")
print(f"Progesterone RRR: {PROG_RRR:.0%} | Screening uptake: {MODEL_SCREENING:.0%} vs {BASELINE_SCREENING:.0%} baseline")
print("="*90)

# Focus on Top 10% threshold
threshold = 'Top 10%'
n_flagged = ptb_data[threshold]['n_flagged']
ppv = ptb_data[threshold]['ppv']

print(f"\nAt {threshold} threshold: n_flagged = {n_flagged:,}, PPV = {ppv:.1%}")
print("-"*90)
print(f"{'Scenario':<45} {'SC Prev':>8} {'Cases Prev':>12} {'Cost/Prev':>15} {'Net Savings':>15} {'Cost-Eff?':>10}")
print("-"*90)

results = []
for scenario_name, sc_prev in SHORT_CERVIX_SCENARIOS.items():
    # Model arm
    n_screened_model = n_flagged * MODEL_SCREENING
    n_short_cervix_model = n_screened_model * sc_prev
    cases_prevented_model = n_short_cervix_model * ppv * PROG_RRR
    cost_model = (n_screened_model * SCREENING_COST) + (n_short_cervix_model * PROGESTERONE_COST)
    
    # Baseline arm
    n_screened_baseline = n_flagged * BASELINE_SCREENING
    n_short_cervix_baseline = n_screened_baseline * sc_prev
    cases_prevented_baseline = n_short_cervix_baseline * ppv * PROG_RRR
    cost_baseline = (n_screened_baseline * SCREENING_COST) + (n_short_cervix_baseline * PROGESTERONE_COST)
    
    # Incremental
    incremental_cases = cases_prevented_model - cases_prevented_baseline
    incremental_cost = cost_model - cost_baseline
    
    if incremental_cases > 0:
        cost_per_prevented = incremental_cost / incremental_cases
        net_savings = PTB_COST - cost_per_prevented
        cost_eff = 'Yes' if net_savings > 0 else 'No'
    else:
        cost_per_prevented = float('inf')
        net_savings = float('-inf')
        cost_eff = 'N/A'
    
    print(f"{scenario_name:<45} {sc_prev:>7.1%} {incremental_cases:>12.2f} ${cost_per_prevented:>14,.0f} ${net_savings:>14,.0f} {cost_eff:>10}")
    
    results.append({
        'Scenario': scenario_name,
        'Short_cervix_prevalence': sc_prev,
        'Cases_prevented': round(incremental_cases, 2),
        'Cost_per_prevented': round(cost_per_prevented, 0),
        'Net_savings': round(net_savings, 0),
        'Cost_effective': cost_eff
    })

df_results = pd.DataFrame(results)
df_results.to_csv('PTB_costeff_numom2b_scenarios.csv', index=False)
print("\nResults saved to PTB_costeff_numom2b_scenarios.csv")
'''

'\n# =============================================================================\n# PTB with NuMom2b Short Cervix Prevalence\n# =============================================================================\n\n# Load PTB metrics from the base_metrics DataFrame already loaded in the first cell\n# (PTB_12w_base_metrics.csv is produced by the Preterm 12 - Clinical Utility notebook).\nptb_data = {\n    row[\'threshold\']: {\'n_flagged\': int(row[\'n_flagged\']), \'ppv\': float(row[\'ppv\'])}\n    for _, row in ptb_metrics.iterrows()\n}\nprint("PTB base metrics loaded for NuMom2b analysis:")\nfor k, v in ptb_data.items():\n    print(f"  {k}: n_flagged={v[\'n_flagged\']:,}, ppv={v[\'ppv\']:.4f}")\n\n# Parameters\nPTB_COST = 51600  # IOM 2007\nSCREENING_COST = 250\nPROGESTERONE_COST = 400\nPROG_RRR = 0.34  # 1 - 0.66\n\n# Screening uptake\nBASELINE_SCREENING = 0.15\nMODEL_SCREENING = 0.90\n\n# NuMom2b-derived short cervix prevalence scenarios\nSHORT_CERVIX_SCENARIOS = {\n    \'Conservative (